# 🏰 Citadel Backend Contracts - Testing Center

## Explore How to Bring New AI Backends/Models into the Citadel Governance Hub!

Use this Jupyter notebook with Python code snippets to onboard all your AI backends and all associated routing logic, including:
- Configuring LLM backend endpoints and authentication
- Generating Terraform variables (`terraform.tfvars`) for deployment
- Deploying backend pools and policy fragments
- Testing deployed models through multiple API formats
- Verifying streaming and SDK integration

> **Note:** This notebook assumes you have already deployed your Citadel Governance Hub. If you haven't done so, please refer to the [Citadel Governance Hub Deployment Guide](../guides/full-deployment-guide.md) or [Citadel Governance Hub Quick Deployment Guide](../guides/quick-deployment-guide.md) before proceeding.

## Azure Prerequisites

- An existing Citadel Governance Hub deployment with APIM configured
- LLM backend endpoints (AI Foundry, Azure OpenAI, or external) provisioned and accessible
- Managed Identity configured for APIM authentication to backend services
- Terraform >= 1.5 and Azure CLI installed and authenticated (`az login`)


<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Start by defining the variables that will drive the configuration of your Access Contracts and the generation of the APIM product policies. Adjust these variables according to your testing needs.

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

inference_api_version = "2024-05-01-preview"

# ============================================================================
# REQUIRED: Update these values for your environment (used as defaults)
# ============================================================================
governance_hub_resource_group = "REPLACE"          # Resource group of the deployed Citadel Governance Hub
location                      = "REPLACE"          # Azure region (e.g. "swedencentral", "eastus")

## ============================================================================
## UPDATE THIS CONFIGURATION BASED ON YOUR DEPLOYED BACKENDS
## This is an example configuration for two Azure AI Foundry backends. Update the values to match your deployed backends and models.

llm_backends_config = [
  {
    "authScheme": "managedIdentity",
    "authType": "managed-identity",
    "backendId": "aif-citadel-primary",
    "backendType": "ai-foundry",
    "endpoint": "https://REPLACE.services.ai.azure.com/",
    "priority": 1,
    "supportedModels": [
      {
        "capacity": 50,
        "modelFormat": "OpenAI",
        "modelVersion": "2026-03-05",
        "name": "gpt-5.4",
        "retirementDate": "2027-03-05",
        "sku": "GlobalStandard",
        "timeout": 180
      },
      {
        "capacity": 1,
        "modelFormat": "OpenAI",
        "modelVersion": "2026-03-17",
        "name": "gpt-5.4-mini",
        "retirementDate": "2027-09-21",
        "sku": "GlobalStandard"
      }
      
    ],
    "weight": 100
  }
]


# ============================================================================
# OPTIONAL: Model Aliases
# ----------------------------------------------------------------------------
# Group multiple models under a single client-facing name. Clients send the
# alias as the `model` value (or `/deployments/{alias}/...` URL); the gateway
# resolves it to a real model. The same alias map is honored consistently
# across Azure OpenAI API, Universal LLM API, and Unified AI API.
#
#   strategy: 'priority'  → first model wins; subsequent models act as
#                           cross-model fallback in the Unified AI API retry.
#             'weighted'  → random selection by weights (A/B testing or
#                           controlled rollouts). Requires `weights`.
#
# Direct model selection is unaffected — when `model` does NOT match an alias,
# routing falls back to the standard model→backend pool lookup.
#
# The defaults below are aligned with the accelerator's main `main.bicepparam`
# model line-up (gpt-4.1, gpt-5.4-mini, gpt-5.2, DeepSeek-R1, Phi-4,
# Mistral-Large-3, text-embedding-3-large). Adjust the `models` lists to
# match the real models you deploy.
# ============================================================================
# model_aliases = [
#     # Most-advanced GPT — automatically falls back when the top model is throttled.
#     { "name": "adv-gpt",        "models": ["gpt-5.4", "gpt-4.1"], "strategy": "priority" },
#     # Cost/latency-optimized GPT — prefers the smaller model with full-size fallback.
#     { "name": "fast-gpt",       "models": ["gpt-4.1-mini"],            "strategy": "priority" },
#     # Reasoning workloads — DeepSeek as primary, GPT-5.2 as fallback.
#     { "name": "reasoning",      "models": ["gpt-5.4"],             "strategy": "priority" },
#     # Stable embeddings alias — easy to swap underlying model later without
#     # touching client code.
#     { "name": "embeddings",     "models": ["text-embedding-3-small"],             "strategy": "priority" },
#     # Weighted A/B blend: 80% to the cheaper model, 20% to the larger one.
#     # Useful to validate gpt-4.1 quality on a slice of production traffic.
#     { "name": "ab-test-gpt",    "models": ["gpt-4.1-mini", "gpt-4.1"],            "strategy": "weighted", "weights": [80, 20] },
# ]
model_aliases = [
  { "name": "adv-gpt",        "models": ["gpt-5.4"], "strategy": "priority" },
  { "name": "mini-gpt",        "models": ["gpt-5.4-mini"], "strategy": "priority" },
  ]  # REPLACE: add your model aliases here, or leave empty to disable

# ============================================================================
# OPTIONAL: Key Vault for backend credential references
# Required when any backend's authConfig uses `keyVaultSecretUri`. The APIM
# managed identity must have `Key Vault Secrets User` role on this vault.
# ============================================================================
key_vault_name = ""

# ============================================================================
# OPTIONAL: AWS credentials (only required when an aws-bedrock backend is used)
# Versionless Key Vault secret URIs; APIM named values aws-access-key / aws-secret-key
# reference them (no plaintext keys). aws-region is a plain named value.
# ============================================================================
aws_access_key_secret_uri = ""  # e.g. https://<kv>.vault.azure.net/secrets/aws-access-key
aws_secret_key_secret_uri = ""  # e.g. https://<kv>.vault.azure.net/secrets/aws-secret-key
aws_region = ""

# Managed Identity for APIM authentication (will be auto-discovered if not specified)
apim_managed_identity_name = ""  # Leave empty to auto-discover


utils.print_info(f"Initialization completed.")


👉🏽 Initialization completed.


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

In [5]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:02:38.414911 :0s]
👉🏽 Current user: nadeemis@MngEnvMCAP507250.onmicrosoft.com
👉🏽 Tenant ID: 1f566f65-6b09-4174-b225-ed68035c8ad5
👉🏽 Subscription ID: c25e83e9-00ef-4f10-8945-a9e113144476


<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected. Initialize the APIM client to interact with your deployment:

In [6]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()
    
    apim_resource_name = apimClientTool.apim_resource_name
    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    
    utils.print_ok(f"APIM Client Tool initialized successfully!")
    utils.print_info(f"APIM Resource Name: {apim_resource_name}")
    utils.print_info(f"APIM Gateway URL: {apim_resource_gateway_url}")
    
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:02:40.962534 :0s]
👉🏽 Current user: nadeemis@MngEnvMCAP507250.onmicrosoft.com
👉🏽 Tenant ID: 1f566f65-6b09-4174-b225-ed68035c8ad5
👉🏽 Subscription ID: c25e83e9-00ef-4f10-8945-a9e113144476
⚙️ Running: az resource list -g rg-citadel-italynorth-01 --resource-type Microsoft.ApiManagement/service 
✅ Listing APIM Resources ⌚ 14:02:44.629058 :3s]
👉🏽 APIM Service Id: /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8
👉🏽 APIM Gateway URL: https://apim-82da66b0c8.azure-api.net
👉🏽 Retrieved key 0 for subscription: master
👉🏽 Retrieved key 1 for subscription: 854bc984-355a-4e87-ae7d-1d0eedab998d
✅ APIM Client Tool initialized successfully! ⌚ 14:02:47.644140 
👉🏽 APIM Resource Name: apim-82da66b0c8
👉🏽 APIM Gateway URL: https://apim-82da66b0c8.azure-api.net


<a id='3'></a>
### 3️⃣ Extract Current APIM Backend-Pools Configuration

Retrieve and analyze the existing backend pools and backends configured in your APIM instance:

In [7]:
# Extract current backends from APIM using the SDK
utils.print_info("Extracting current APIM backends configuration...")

try:
    # Use the APIMClientTool's new get_backends method (uses Azure SDK instead of CLI)
    existing_backends, existing_backend_pools = apimClientTool.get_backends()
    
except Exception as e:
    utils.print_error(f"Error extracting backends: {e}")
    existing_backends = []
    existing_backend_pools = []

👉🏽 Extracting current APIM backends configuration...
👉🏽 Retrieving APIM backends using Azure REST API...
👉🏽 🔗 Backend: content-safety-backend -> https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
👉🏽 🔗 Backend: foundry-italynorth-0 -> https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
👉🏽 🔗 Backend: ms-learn-mcp-backend -> https://learn.microsoft.com/api/mcp
✅ Found 3 individual backends and 0 backend pools ⌚ 14:25:25.585936 


In [8]:
# Get supported models from the policy fragment (if exists)
try:
    supported_models_from_policy = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Supported models in APIM policy fragment 'set-backend-pools':")
    for model in supported_models_from_policy:
        print(f"  • {model}")
except Exception as e:
    utils.print_warning(f"Could not retrieve policy fragment (may not exist yet): {e}")
    supported_models_from_policy = []

👉🏽 Retrieved policy fragment: set-backend-pools
👉🏽 Found 2 unique supported models
✅ Supported models in APIM policy fragment 'set-backend-pools': ⌚ 14:25:32.342173 
  • gpt-5.4
  • gpt-5.4-mini


In [9]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

👉🏽 
👉🏽 CURRENT APIM BACKEND CONFIGURATION SUMMARY
👉🏽 ============================================================

📋 Individual Backends:
  • content-safety-backend
    URL: https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
  • foundry-italynorth-0
    URL: https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
    Models: gpt-5.4, gpt-5.4-mini
  • ms-learn-mcp-backend
    URL: https://learn.microsoft.com/api/mcp

🤖 Total Supported Models: 2
   gpt-5.4, gpt-5.4-mini


<a id='4'></a>
### 4️⃣ Discover Managed Identity for APIM Authentication

Auto-discover or specify the user-assigned managed identity used by APIM:

In [10]:
# Discover managed identity from APIM using the SDK
utils.print_info("Discovering managed identity configuration...")

# Use the APIMClientTool's get_managed_identity_info method
managed_identity_info = apimClientTool.get_managed_identity_info()

managed_identity_client_id = managed_identity_info.get('clientId')
managed_identity_name = managed_identity_info.get('name') or apim_managed_identity_name
managed_identity_resource_group = managed_identity_info.get('resourceGroup') or governance_hub_resource_group

if not managed_identity_client_id:
    utils.print_warning("Could not auto-discover managed identity. Please specify it manually in the configuration.")
else:
    utils.print_info(f"Client ID: {managed_identity_client_id}")

if managed_identity_name:
    utils.print_ok(f"Managed Identity Name: {managed_identity_name}")
    utils.print_ok(f"Managed Identity Resource Group: {managed_identity_resource_group}")

👉🏽 Discovering managed identity configuration...
✅ Found managed identity client ID in named values: e226920a... ⌚ 14:25:49.040280 
✅ Found user-assigned managed identity: id-apim-citadel-italynorth-01-uv5wjw ⌚ 14:25:49.040385 
👉🏽 Client ID: e226920a-fe36-4353-98b7-9efa236f32cc
✅ Managed Identity Name: id-apim-citadel-italynorth-01-uv5wjw ⌚ 14:25:49.040600 
✅ Managed Identity Resource Group: rg-citadel-italynorth-01 ⌚ 14:25:49.040609 


<a id='5'></a>
### 5️⃣ Generate LLM Backend Terraform Variables File

Generate a customizable `terraform.tfvars` file with the full list of LLM backends to be integrated with APIM. This file feeds the standalone Terraform module at [`../llm-backend-onboarding`](../llm-backend-onboarding/README.md):


In [11]:
# Configure the LLM backends for deployment
# You can modify the llm_backends_config list defined in the initialization cell

utils.print_info("LLM Backends to be deployed:")
for backend in llm_backends_config:
    print(f"\n  🔗 {backend['backendId']}")
    print(f"     Type: {backend['backendType']}")
    print(f"     Endpoint: {backend['endpoint']}")
    auth = backend.get('authType') or backend.get('authScheme') or 'unspecified'
    print(f"     Auth: {auth}")
    auth_config = backend.get('authConfig')
    if auth_config:
        nv = auth_config.get('namedValueKey', 'N/A')
        if auth_config.get('keyVaultSecretUri'):
            print(f"     Auth Source: Key Vault → {auth_config['keyVaultSecretUri']} (named value: {nv})")
        elif auth_config.get('secretValue'):
            print(f"     Auth Source: inline secret (named value: {nv}) ⚠️ testing only")
        else:
            print(f"     Auth Source: named value '{nv}'")
    print(f"     Priority: {backend.get('priority', 1)}, Weight: {backend.get('weight', 100)}")
    # Display supported models with their per-model metadata
    print(f"     Models ({len(backend['supportedModels'])}):")
    for model in backend['supportedModels']:
        model_name = model['name'] if isinstance(model, dict) else model
        if isinstance(model, dict):
            sku = model.get('sku', 'Standard')
            capacity = model.get('capacity', 100)
            fmt = model.get('modelFormat', 'OpenAI')
            ver = model.get('modelVersion', '1')
            extras = []
            if 'retirementDate' in model:
                extras.append(f"Retirement: {model['retirementDate']}")
            if 'apiVersion' in model:
                extras.append(f"API: {model['apiVersion']}")
            if 'timeout' in model:
                extras.append(f"Timeout: {model['timeout']}s")
            if 'inferenceApiVersion' in model:
                extras.append(f"Inference API: {model['inferenceApiVersion']}")
            extra_str = f", {', '.join(extras)}" if extras else ""
            print(f"       - {model_name} (SKU: {sku}, Capacity: {capacity}, Format: {fmt}, Version: {ver}{extra_str})")
        else:
            print(f"       - {model_name}")

# Display model aliases if defined
if model_aliases:
    utils.print_info(f"\nModel Aliases ({len(model_aliases)}):")
    for alias in model_aliases:
        strategy = alias.get('strategy', 'priority')
        models_str = ', '.join(alias.get('models', []))
        weights = alias.get('weights')
        weights_str = f" weights={weights}" if weights else ""
        print(f"  • {alias['name']} → [{models_str}]  (strategy: {strategy}{weights_str})")


👉🏽 LLM Backends to be deployed:

  🔗 aif-citadel-primary
     Type: ai-foundry
     Endpoint: https://aif-citadel-italynorth-01-0-uv5wjw.services.ai.azure.com/
     Auth: managed-identity
     Priority: 1, Weight: 100
     Models (2):
       - gpt-5.4 (SKU: GlobalStandard, Capacity: 50, Format: OpenAI, Version: 2026-03-05, Retirement: 2027-03-05, Timeout: 180s)
       - gpt-5.4-mini (SKU: GlobalStandard, Capacity: 1, Format: OpenAI, Version: 2026-03-17, Retirement: 2027-09-21)
👉🏽 
Model Aliases (2):
  • adv-gpt → [gpt-5.4]  (strategy: priority)
  • mini-gpt → [gpt-5.4-mini]  (strategy: priority)


In [12]:
# Generate the terraform.tfvars file for the standalone llm-backend-onboarding Terraform module
# (replaces the legacy .bicepparam generation — the deployment now runs `terraform apply`)

# Path to the Terraform module (relative to this notebook in validation/)
tf_dir = os.path.abspath("../llm-backend-onboarding")
if not os.path.isdir(tf_dir):
    utils.print_error(f"Terraform module directory not found: {tf_dir}")
tfvars_file = os.path.join(tf_dir, "llm-backends-generated-local.tfvars")


def hcl_str(v):
    """Quote and escape a string value for HCL."""
    return '"' + str(v).replace("\\", "\\\\").replace('"', '\\"') + '"'

# Format a single model object for Terraform (HCL).
# Model field names stay camelCase to match the Terraform variable schema
# (name, sku, capacity, modelFormat, modelVersion, ...).
def format_model_for_tf(model):
    if isinstance(model, str):
        # Legacy format: just model name string - convert to object with defaults
        return f"{{ name = {hcl_str(model)} }}"

    parts = [f"name = {hcl_str(model['name'])}"]
    if 'sku' in model:                 parts.append(f"sku = {hcl_str(model['sku'])}")
    if 'capacity' in model:            parts.append(f"capacity = {model['capacity']}")
    if 'modelFormat' in model:         parts.append(f"modelFormat = {hcl_str(model['modelFormat'])}")
    if 'modelVersion' in model:        parts.append(f"modelVersion = {hcl_str(model['modelVersion'])}")
    if 'retirementDate' in model:      parts.append(f"retirementDate = {hcl_str(model['retirementDate'])}")
    if 'apiVersion' in model:          parts.append(f"apiVersion = {hcl_str(model['apiVersion'])}")
    if 'timeout' in model:             parts.append(f"timeout = {model['timeout']}")
    if 'inferenceApiVersion' in model: parts.append(f"inferenceApiVersion = {hcl_str(model['inferenceApiVersion'])}")
    return "{ " + ", ".join(parts) + " }"

# Map the notebook's camelCase authConfig keys to the Terraform snake_case schema.
def format_auth_config_for_tf(auth_config):
    parts = []
    if auth_config.get('namedValueKey'):
        parts.append(f"named_value_key = {hcl_str(auth_config['namedValueKey'])}")
    if auth_config.get('keyVaultSecretUri'):
        parts.append(f"key_vault_secret_uri = {hcl_str(auth_config['keyVaultSecretUri'])}")
    if auth_config.get('secretValue'):
        parts.append(f"secret_value = {hcl_str(auth_config['secretValue'])}")
    return "{ " + ", ".join(parts) + " }"

# Format a backend object for Terraform (snake_case keys, per-model metadata).
def format_backend_for_tf(backend):
    models_formatted = [format_model_for_tf(m) for m in backend['supportedModels']]
    # NOTE: HCL list items MUST be comma-separated. Join with a comma so the
    #       generated supported_models = [ ... ] block is valid HCL.
    models_str = ",\n      ".join(models_formatted)

    lines = [
        "  {",
        f"    backend_id   = {hcl_str(backend['backendId'])}",
        f"    backend_type = {hcl_str(backend['backendType'])}",
        f"    endpoint     = {hcl_str(backend['endpoint'])}",
    ]
    # Auth: carry over legacy authScheme and/or new authType
    if 'authScheme' in backend:
        lines.append(f"    auth_scheme  = {hcl_str(backend['authScheme'])}")
    if 'authType' in backend:
        lines.append(f"    auth_type    = {hcl_str(backend['authType'])}")
    if backend.get('authConfig'):
        lines.append(f"    auth_config  = {format_auth_config_for_tf(backend['authConfig'])}")
    lines.append(f"    supported_models = [\n      {models_str}\n    ]")
    lines.append(f"    priority = {backend.get('priority', 1)}")
    lines.append(f"    weight   = {backend.get('weight', 100)}")
    lines.append("  }")
    return "\n".join(lines)

# Format a model alias object for Terraform.
def format_alias_for_tf(alias):
    parts = [f"name = {hcl_str(alias['name'])}"]
    models_arr = ", ".join([hcl_str(m) for m in alias.get('models', [])])
    parts.append(f"models = [ {models_arr} ]")
    if 'strategy' in alias:
        parts.append(f"strategy = {hcl_str(alias['strategy'])}")
    if 'weights' in alias:
        weights_arr = ", ".join([str(w) for w in alias['weights']])
        parts.append(f"weights = [ {weights_arr} ]")
    return "  { " + ", ".join(parts) + " }"


# NOTE: HCL list items MUST be comma-separated. Join backends with a comma too.
backends_tf_str = ",\n".join([format_backend_for_tf(b) for b in llm_backends_config])

# Model aliases block
if model_aliases:
    aliases_str = ",\n".join([format_alias_for_tf(a) for a in model_aliases])
    aliases_block = f"model_aliases = [\n{aliases_str}\n]\n"
else:
    aliases_block = "model_aliases = []\n"

# Optional Key Vault + AWS credential blocks
optional_block = ""
if key_vault_name:
    optional_block += f"\n# Key Vault for backend credential references (authConfig.keyVaultSecretUri)\n"
    optional_block += f"key_vault_name = {hcl_str(key_vault_name)}\n"
if aws_access_key_secret_uri or aws_secret_key_secret_uri or aws_region:
    optional_block += f"\n# AWS credentials for Amazon Bedrock (SigV4) backends\n"
    optional_block += f"aws_access_key_secret_uri = {hcl_str(aws_access_key_secret_uri)}\n"
    optional_block += f"aws_secret_key_secret_uri = {hcl_str(aws_secret_key_secret_uri)}\n"
    optional_block += f"aws_region                = {hcl_str(aws_region)}\n"

tfvars_content = f"""# ============================================================================
# LLM Backend Onboarding - Generated Terraform Variables
# Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}
# Module:    {tf_dir}
# ============================================================================

# ============================================================================
# Azure Subscription & APIM Details
# ============================================================================
subscription_id     = {hcl_str(subscription_id)}
resource_group_name = {hcl_str(governance_hub_resource_group)}
apim_name           = {hcl_str(apim_resource_name)}

# ============================================================================
# APIM User-Assigned Managed Identity (client ID used for backend auth)
# ============================================================================
managed_identity_client_id = {hcl_str(managed_identity_client_id or '')}

# ============================================================================
# Circuit Breaker (recommended for production)
# ============================================================================
configure_circuit_breaker = true

# ============================================================================
# LLM Backend Configuration Array
# Each model in supported_models has its own metadata (sku, capacity,
#   modelFormat, modelVersion, retirementDate, apiVersion, timeout,
#   inferenceApiVersion).
# ============================================================================
llm_backend_config = [
{backends_tf_str}
]

# ============================================================================
# Model Aliases (group multiple models under a single client-facing name)
# ============================================================================
{aliases_block}{optional_block}"""

# Write the variables file
utils.print_info(f"Generating Terraform variables file: {tfvars_file}")
with open(tfvars_file, 'w') as f:
    f.write(tfvars_content)

utils.print_ok("Terraform variables file generated successfully!")
print("\n" + "="*60)
print("GENERATED terraform.tfvars CONTENT:")
print("="*60)
print(tfvars_content)


👉🏽 Generating Terraform variables file: /Users/nadeemis/Projects/terraform-ai-hub-gateway-landing-zone/llm-backend-onboarding/llm-backends-generated-local.tfvars
✅ Terraform variables file generated successfully! ⌚ 14:26:55.802598 

GENERATED terraform.tfvars CONTENT:
# ============================================================================
# LLM Backend Onboarding - Generated Terraform Variables
# Generated: 2026-10-07 14:26:55
# Module:    /Users/nadeemis/Projects/terraform-ai-hub-gateway-landing-zone/llm-backend-onboarding
# ============================================================================

# ============================================================================
# Azure Subscription & APIM Details
# ============================================================================
subscription_id     = "c25e83e9-00ef-4f10-8945-a9e113144476"
resource_group_name = "rg-citadel-italynorth-01"
apim_name           = "apim-82da66b0c8"

# ====================================

<a id='6'></a>
### 6️⃣ Deploy LLM Backend Onboarding (Terraform)

Deploy the LLM backends, backend pools, and policy fragments to APIM by running the Terraform commands with the generated variables file. The script runs `terraform init`, imports any pre-existing resources, then `terraform apply`:


In [13]:
# ============================================================================
# 6️⃣ Deploy with Terraform commands — does not use bash helper script
# ----------------------------------------------------------------------------
# This cell reproduces everything llm-backend-onboarding/scripts/deploy.sh + llm-backend-onboarding/scripts/import-existing.sh
# do, but by invoking `terraform` and `az` directly. Commands are run via
# argument LISTS (shell=False), so:
#   • Resource addresses containing quotes/brackets need no shell escaping.
#   • There is no dependency on bash / Git Bash / WSL.
#   • `az` (which is az.cmd on Windows) is launched correctly via cmd /c.
#
# Prerequisites already set by earlier cells:
#   tf_dir, tfvars_file, subscription_id, governance_hub_resource_group,
#   apim_resource_name, llm_backends_config
# ============================================================================
import subprocess, shutil, json

assert os.path.isdir(tf_dir),       f"Terraform module dir not found: {tf_dir}"
assert os.path.isfile(tfvars_file), f"tfvars file not found: {tfvars_file}"


def _resolve_exe(name):
    """Resolve an executable on PATH (finds terraform.exe / az.cmd on Windows)."""
    return shutil.which(name) or name


TERRAFORM_EXE = _resolve_exe("terraform")
AZ_EXE        = _resolve_exe("az")


def _exec(args, cwd=None, quiet=False):
    """Run a command given as a list of args, without a shell.

    On Windows, batch wrappers (az.cmd) must be launched through cmd.exe, so we
    transparently prefix `cmd /c` for .cmd/.bat targets.
    """
    run_args = args
    if os.name == "nt" and args[0].lower().endswith((".cmd", ".bat")):
        run_args = ["cmd", "/c", *args]
    if not quiet:
        utils.print_info("⚙️  " + " ".join(args))
    return subprocess.run(
        run_args, cwd=cwd, capture_output=True, text=True,
        encoding="utf-8", errors="replace",
    )


def tf(*args, quiet=False):
    """Run a terraform subcommand inside the module directory."""
    return _exec([TERRAFORM_EXE, *args], cwd=tf_dir, quiet=quiet)


# ----------------------------------------------------------------------------
# 1) terraform init
# ----------------------------------------------------------------------------
utils.print_info("Initializing Terraform...")
r = tf("init", "-upgrade")
print(r.stdout + r.stderr)
if r.returncode != 0:
    raise RuntimeError("terraform init failed — see output above.")
utils.print_ok("terraform init complete")

# ----------------------------------------------------------------------------
# 2) Import pre-existing Azure resources into state (replaces import-existing.sh)
#    Resources created by the main Citadel deployment are imported so Terraform
#    updates them in place instead of failing with "already exists".
# ----------------------------------------------------------------------------
APIM_BASE_ID = (
    f"/subscriptions/{subscription_id}"
    f"/resourceGroups/{governance_hub_resource_group}"
    f"/providers/Microsoft.ApiManagement/service/{apim_resource_name}"
)
ARM_API_VERSION = "2024-06-01-preview"

_imported = _skipped = _not_found = 0


def try_import(tf_address, azure_id):
    """Import an Azure resource only if it isn't already in state but exists in Azure."""
    global _imported, _skipped, _not_found
    # Already tracked in Terraform state?
    if tf("state", "show", tf_address, quiet=True).returncode == 0:
        _skipped += 1
        return
    # Does it actually exist in Azure?
    az_url = f"https://management.azure.com{azure_id}?api-version={ARM_API_VERSION}"
    if _exec([AZ_EXE, "rest", "--method", "GET", "--url", az_url], quiet=True).returncode != 0:
        _not_found += 1
        return
    utils.print_info(f"  Importing {tf_address}")
    res = tf("import", f"-var-file={tfvars_file}", tf_address, azure_id, quiet=True)
    if res.returncode == 0:
        utils.print_ok(f"  Imported {tf_address}")
        _imported += 1
    else:
        utils.print_warning(
            f"  Could not import {tf_address} (will be recreated): {(res.stderr or '').strip()[:200]}"
        )


utils.print_info(f"Checking for existing resources to import (APIM: {apim_resource_name})...")

# -- LLM backends (from llm_backends_config in memory) --
for _b in llm_backends_config:
    _bid = _b["backendId"]
    try_import(f'azapi_resource.llm_backend["{_bid}"]', f"{APIM_BASE_ID}/backends/{_bid}")

# -- Dynamic policy fragments (TF resource name != fragment id) --
_dynamic_fragments = {
    "set-backend-pools":    "azurerm_api_management_policy_fragment.set_backend_pools",
    "get-available-models": "azurerm_api_management_policy_fragment.get_available_models",
    "metadata-config":      "azurerm_api_management_policy_fragment.metadata_config",
}
for _frag_id, _addr in _dynamic_fragments.items():
    try_import(_addr, f"{APIM_BASE_ID}/policyFragments/{_frag_id}")

# -- Static policy fragments (for_each map keyed by fragment id) --
_static_fragments = [
    "set-backend-authorization", "set-target-backend-pool", "set-llm-requested-model",
    "set-llm-usage", "validate-model-access", "responses-id-security", "responses-id-cache-store",
]
for _frag in _static_fragments:
    try_import(
        f'azurerm_api_management_policy_fragment.static["{_frag}"]',
        f"{APIM_BASE_ID}/policyFragments/{_frag}",
    )

# -- resolve-model-alias fragment (its own resource block) --
try_import(
    "azurerm_api_management_policy_fragment.resolve_model_alias",
    f"{APIM_BASE_ID}/policyFragments/resolve-model-alias",
)

# -- AWS Bedrock named values (always created with safe defaults) --
_aws_named_values = {
    "aws-access-key": "azurerm_api_management_named_value.aws_access_key",
    "aws-secret-key": "azurerm_api_management_named_value.aws_secret_key",
    "aws-region":     "azurerm_api_management_named_value.aws_region",
}
for _nv_id, _addr in _aws_named_values.items():
    try_import(_addr, f"{APIM_BASE_ID}/namedValues/{_nv_id}")

# -- Per-backend API-key named values (auth_config.named_value_key) --
for _b in llm_backends_config:
    _nv_key = (_b.get("authConfig") or {}).get("namedValueKey")
    if _nv_key:
        try_import(
            f'azurerm_api_management_named_value.backend_api_key["{_nv_key}"]',
            f"{APIM_BASE_ID}/namedValues/{_nv_key}",
        )

# -- Existing backend pools (discovered via ARM) --
_pools_res = _exec(
    [AZ_EXE, "rest", "--method", "GET",
     "--url", f"https://management.azure.com{APIM_BASE_ID}/backends?api-version={ARM_API_VERSION}"],
    quiet=True,
)
if _pools_res.returncode == 0:
    try:
        for _be in json.loads(_pools_res.stdout).get("value", []):
            if _be.get("properties", {}).get("type") == "Pool":
                _pname = _be["name"]
                try_import(
                    f'azapi_resource.llm_backend_pool["{_pname}"]',
                    f"{APIM_BASE_ID}/backends/{_pname}",
                )
    except Exception as _e:
        utils.print_warning(f"Could not enumerate existing backend pools: {_e}")

if _imported:
    utils.print_ok(f"Import complete: {_imported} imported, {_skipped} already in state, {_not_found} not in Azure.")
else:
    utils.print_info(f"No imports needed: {_skipped} already in state, {_not_found} not in Azure.")

# ----------------------------------------------------------------------------
# 3) terraform plan
# ----------------------------------------------------------------------------
utils.print_info("Planning deployment...")
r = tf("plan", f"-var-file={tfvars_file}", "-out=tfplan")
print(r.stdout + r.stderr)
if r.returncode != 0:
    raise RuntimeError("terraform plan failed — see output above.")
utils.print_ok("terraform plan complete")

# ----------------------------------------------------------------------------
# 4) terraform apply (applies the saved plan; no prompt)
# ----------------------------------------------------------------------------
utils.print_info("Applying deployment...")
r = tf("apply", "-auto-approve", "tfplan")
print(r.stdout + r.stderr)
if r.returncode != 0:
    raise RuntimeError("terraform apply failed — see output above.")
utils.print_ok("LLM backend onboarding deployment completed successfully!")

# ----------------------------------------------------------------------------
# 5) terraform output
# ----------------------------------------------------------------------------
_out = tf("output", "-json", quiet=True)
if _out.returncode == 0 and _out.stdout.strip():
    try:
        _outputs = json.loads(_out.stdout)
        print("\n" + "=" * 60)
        print("TERRAFORM OUTPUTS:")
        print("=" * 60)
        for _key, _meta in _outputs.items():
            print(f"  {_key}: {_meta.get('value')}")
    except Exception as _e:
        utils.print_warning(f"Could not parse terraform outputs: {_e}")
else:
    utils.print_warning("Could not retrieve terraform outputs.")


👉🏽 Initializing Terraform...
👉🏽 ⚙️  /opt/homebrew/bin/terraform init -upgrade
Initializing the backend...

Initializing provider plugins...
- Finding hashicorp/azurerm versions matching "~> 4.0"...
- Finding azure/azapi versions matching "~> 2.0"...
- Installing hashicorp/azurerm v4.81.0...
- Installed hashicorp/azurerm v4.81.0 (signed by HashiCorp)
- Installing azure/azapi v2.13.0...
- Installed azure/azapi v2.13.0 (signed by a HashiCorp partner, key ID 6F0B91BDE98478CF)
Partner and community providers are signed by their developers.
If you'd like to know more about provider signing, you can read about it here:
https://developer.hashicorp.com/terraform/cli/plugins/signing

Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!

<a id='7'></a>
### 7️⃣ Verify Deployed Configuration

Verify that the backends, pools, and policy fragments were created successfully:

In [17]:
# Re-initialize APIM client to pick up new backends
apimClientTool.initialize()

# Get updated supported models from policy fragment
try:
    updated_supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Updated supported models in APIM policy fragment 'set-backend-pools':")
    for model in updated_supported_models:
        print(f"  • {model}")
except Exception as e:
    utils.print_error(f"Error retrieving policy fragment: {e}")
    updated_supported_models = []

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:45:18.297439 :0s]
👉🏽 Current user: nadeemis@MngEnvMCAP507250.onmicrosoft.com
👉🏽 Tenant ID: 1f566f65-6b09-4174-b225-ed68035c8ad5
👉🏽 Subscription ID: c25e83e9-00ef-4f10-8945-a9e113144476
👉🏽 APIM Service Id: /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8
👉🏽 APIM Gateway URL: https://apim-82da66b0c8.azure-api.net
👉🏽 Retrieved key 0 for subscription: master
👉🏽 Retrieved key 1 for subscription: 854bc984-355a-4e87-ae7d-1d0eedab998d
👉🏽 Retrieved policy fragment: set-backend-pools
👉🏽 Found 2 unique supported models
✅ Updated supported models in APIM policy fragment 'set-backend-pools': ⌚ 14:45:21.282428 
  • gpt-5.4
  • gpt-5.4-mini


In [18]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

👉🏽 
👉🏽 CURRENT APIM BACKEND CONFIGURATION SUMMARY
👉🏽 ============================================================

📋 Individual Backends:
  • content-safety-backend
    URL: https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
  • foundry-italynorth-0
    URL: https://aif-citadel-italynorth-01-0-uv5wjw.cognitiveservices.azure.com/
    Models: gpt-5.4, gpt-5.4-mini
  • ms-learn-mcp-backend
    URL: https://learn.microsoft.com/api/mcp

🤖 Total Supported Models: 2
   gpt-5.4, gpt-5.4-mini


<a id='8'></a>
### 8️⃣ Verify Get Available Models Policy Fragment

Verify that the new `get-available-models` policy fragment was created successfully:

In [19]:
# Verify the get-available-models policy fragment exists
try:
    # Use the Azure SDK to check if the policy fragment exists
    policy_fragment = apimClientTool.client.policy_fragment.get(
        resource_group_name=apimClientTool.resource_group_name,
        service_name=apimClientTool.apim_resource_name,
        id="get-available-models"
    )
    
    if policy_fragment:
        utils.print_ok("Policy fragment 'get-available-models' exists!")
        utils.print_info("This fragment returns available model deployments in a format similar to Azure Cognitive Services API.")
        utils.print_info(f"Description: {policy_fragment.description}")
except Exception as e:
    utils.print_warning(f"Could not retrieve 'get-available-models' policy fragment: {e}")
    utils.print_info("This fragment will be created after running the deployment.")

✅ Policy fragment 'get-available-models' exists! ⌚ 14:46:57.203117 
👉🏽 This fragment returns available model deployments in a format similar to Azure Cognitive Services API.
👉🏽 Description: Returns available model deployments


<a id='test-foundry'></a>
### 🧪 Test GET /deployments (Microsoft Foundry Integration)

Test the `GET /deployments` endpoint which leverages the `get-available-models` policy fragment. This endpoint is used by Microsoft Foundry to discover available model deployments:

In [20]:
# Test GET /deployments endpoint (Microsoft Foundry integration)
# This endpoint uses the get-available-models policy fragment to return available model deployments
# Testing both Universal LLM API and Azure OpenAI API endpoints

def test_get_deployments(base_endpoint, api_name, api_key):
    """Test GET /deployments for a given API endpoint."""
    deployments_url = f"{base_endpoint}deployments?api-version={inference_api_version}"
    utils.print_info(f"\nTesting {api_name} GET /deployments: {deployments_url}")
    
    try:
        response = requests.get(
            deployments_url,
            headers={"api-key": api_key},
            timeout=30
        )
        
        utils.print_response_code(response)
        
        if response.status_code == 200:
            data = response.json()
            deployments = data.get("value", [])
            
            utils.print_ok(f"{api_name} GET /deployments returned {len(deployments)} model deployment(s)")
            
            print("\n" + "="*60)
            print(f"AVAILABLE MODEL DEPLOYMENTS ({api_name})")
            print("="*60)
            
            for deployment in deployments:
                print(f"\n📦 Deployment: {deployment.get('name', 'N/A')}")
                print(f"   ID: {deployment.get('id', 'N/A')}")
                print(f"   Type: {deployment.get('type', 'N/A')}")
                
                sku = deployment.get('sku', {})
                if sku:
                    print(f"   SKU: {sku.get('name', 'N/A')} (Capacity: {sku.get('capacity', 'N/A')})")
                
                props = deployment.get('properties', {})
                if props:
                    model = props.get('model', {})
                    if model:
                        print(f"   Model: {model.get('name', 'N/A')} (Format: {model.get('format', 'N/A')}, Version: {model.get('version', 'N/A')})")
                    
                    capabilities = props.get('capabilities', {})
                    if capabilities:
                        caps_list = [k for k, v in capabilities.items() if v == 'true' or v == True]
                        print(f"   Capabilities: {', '.join(caps_list) if caps_list else 'N/A'}")
                    
                    print(f"   Status: {props.get('provisioningState', 'N/A')}")
            
            print("\n" + "="*60)
            utils.print_ok(f"{api_name} GET /deployments test completed successfully!")
            return True
        else:
            utils.print_error(f"{api_name} GET /deployments failed: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except Exception as e:
        utils.print_error(f"{api_name} GET /deployments test failed: {str(e)}")
        return False

# Get an API key from subscriptions
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[0].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[0].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

if api_key:
    # Test 1: Universal LLM API - GET /models/deployments
    apimClientTool.discover_api("models")
    azure_endpoint_models = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Universal LLM API Base Endpoint: {azure_endpoint_models}models")
    test_get_deployments(f"{azure_endpoint_models}models/", "Universal LLM API", api_key)
    
    # Test 2: Azure OpenAI API - GET /openai/deployments
    try:
        apimClientTool.discover_api("openai")
        azure_endpoint_openai = str(apimClientTool.azure_endpoint)
        utils.print_info(f"\nAzure OpenAI API Base Endpoint: {azure_endpoint_openai}openai")
        test_get_deployments(f"{azure_endpoint_openai}openai/", "Azure OpenAI API", api_key)
    except Exception as e:
        utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
else:
    utils.print_warning("Cannot test GET /deployments - missing API key")

✅ Using subscription: master ⌚ 14:47:11.506908 
👉🏽 Found API with id /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8/apis/universal-llm-api and path models
👉🏽 Azure Endpoint with APIM https://apim-82da66b0c8.azure-api.net/
👉🏽 Universal LLM API Base Endpoint: https://apim-82da66b0c8.azure-api.net/models
👉🏽 
Testing Universal LLM API GET /deployments: https://apim-82da66b0c8.azure-api.net/models/deployments?api-version=2024-05-01-preview
Response status: 200 - OK
✅ Universal LLM API GET /deployments returned 4 model deployment(s) ⌚ 14:47:15.618109 

AVAILABLE MODEL DEPLOYMENTS (Universal LLM API)

📦 Deployment: gpt-5.4
   ID: aif-citadel-primary
   Type: ai-foundry
   SKU: GlobalStandard (Capacity: 50)
   Model: gpt-5.4 (Format: OpenAI, Version: 2026-03-05)
   Capabilities: chatCompletion
   Status: Succeeded

📦 Deployment: gpt-5.4-mini
   ID: aif-citadel-primary
   Type: ai-foundry
   S

---
## 🧪 Test Deployed Models

The following sections test the deployed models through both the Universal LLM API and Azure OpenAI API endpoints.
---

<a id='test-universal'></a>
### 🧪 Test via Universal LLM API (models/chat/completions)

Test the deployed models using the Universal LLM API which routes based on the `model` field in the request body:

> **Note:** Some models may not support the chat/completions format. Please refer to your LLM backend documentation for supported API formats per model. There is a validation notebook that is designed to test both chat/completions, embeddings, and response formats.

In [21]:
# Discover the Universal LLM API endpoint
apimClientTool.discover_api("models")
azure_endpoint_models = str(apimClientTool.azure_endpoint)
chat_completions_url_models = f"{azure_endpoint_models}models/chat/completions?api-version={inference_api_version}"

utils.print_info(f"Universal LLM API Endpoint: {chat_completions_url_models}")

# Get an API key from subscriptions (it will get the most recently created Access Contract which should be the one we created for testing)
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[0].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[0].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

👉🏽 Found API with id /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8/apis/universal-llm-api and path models
👉🏽 Azure Endpoint with APIM https://apim-82da66b0c8.azure-api.net/
👉🏽 Universal LLM API Endpoint: https://apim-82da66b0c8.azure-api.net/models/chat/completions?api-version=2024-05-01-preview
✅ Using subscription: master ⌚ 14:47:35.164087 


In [22]:
# Test each supported model via Universal LLM API
# NOTE: This loop sends a chat/completions request to every model in `updated_supported_models`.
#       Embedding-only models (e.g. text-embedding-*) cannot serve chat/completions and the
#       backend correctly returns HTTP 400 with a body like:
#           "The requested operation is unsupported with the model selected"
#       That outcome proves the request was routed and the backend rejected on its merits, so
#       we treat it as a SUCCESSFUL routing test (not a failure).
def is_unsupported_operation_400(response):
    """Return True for backend 400s that indicate the model doesn't support this API surface
    (typical when chat/completions is sent to an embedding model)."""
    if response.status_code != 400:
        return False
    text = (response.text or "").lower()
    markers = (
        "operation is unsupported",
        "operation_not_supported",
        "operationnotsupported",
        "unsupportedoperation",
        "not supported with the model",
    )
    return any(m in text for m in markers)

if api_key and updated_supported_models:
    utils.print_info(f"\nTesting {len(updated_supported_models)} models via Universal LLM API...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is 2+2? Answer in one word."}
    ]
    
    for model_name in updated_supported_models:  # add [:3]:  to test first 3 models only
        utils.print_info(f"Testing model: {model_name}")
        
        payload = {
            "model": model_name,
            "messages": test_messages
        }
        
        try:
            utils.print_info(f"Using endpoint: {chat_completions_url_models}")
            
            response = requests.post(
                chat_completions_url_models,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            elif is_unsupported_operation_400(response):
                # Expected for embedding-only models invoked via chat/completions.
                print(f"  ℹ️  Backend returned 400 'operation unsupported' — expected for embedding-only models on chat/completions.")
                utils.print_ok(f"Model '{model_name}' - SUCCESS (routed; chat/completions not supported by this model)\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run tests - missing API key or supported models")


👉🏽 
Testing 2 models via Universal LLM API...

👉🏽 Testing model: gpt-5.4
👉🏽 Using endpoint: https://apim-82da66b0c8.azure-api.net/models/chat/completions?api-version=2024-05-01-preview
Response status: 200 - OK
  💬 Response: 4
  📍 Backend Region: Italy North
✅ Model 'gpt-5.4' - SUCCESS
 ⌚ 14:47:52.163684 
👉🏽 Testing model: gpt-5.4-mini
👉🏽 Using endpoint: https://apim-82da66b0c8.azure-api.net/models/chat/completions?api-version=2024-05-01-preview
Response status: 200 - OK
  💬 Response: Four
  📍 Backend Region: Italy North
✅ Model 'gpt-5.4-mini' - SUCCESS
 ⌚ 14:47:54.306366 


<a id='test-openai'></a>
### 🧪 Test via Azure OpenAI API (openai/deployments/{model}/chat/completions)

Test the deployed models using the Azure OpenAI compatible API which uses the deployment name in the URL path:

In [23]:
# Discover the Azure OpenAI API endpoint
try:
    apimClientTool.discover_api("openai")
    azure_endpoint_openai = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Azure OpenAI API Base Endpoint: {azure_endpoint_openai}")
except Exception as e:
    utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
    azure_endpoint_openai = None

👉🏽 Found API with id /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8/apis/azure-openai-api and path openai
👉🏽 Azure Endpoint with APIM https://apim-82da66b0c8.azure-api.net/
👉🏽 Azure OpenAI API Base Endpoint: https://apim-82da66b0c8.azure-api.net/


In [24]:
# Test models via Azure OpenAI API format
# NOTE: Embedding-only models will return HTTP 400 "operation is unsupported" when invoked
#       through chat/completions. We treat that as a SUCCESSFUL routing test — the helper
#       `is_unsupported_operation_400` (defined in the Universal LLM API test cell above) is
#       reused here.
if api_key and azure_endpoint_openai and updated_supported_models:
    utils.print_info(f"\nTesting models via Azure OpenAI API format...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is the capital of France? Answer in one word."}
    ]
    
    for model_name in updated_supported_models:  # add [:3]:  to test first 3 models only
        utils.print_info(f"Testing model: {model_name}")
        
        # Azure OpenAI format uses deployment name in URL path
        chat_completions_url_openai = f"{azure_endpoint_openai}openai/deployments/{model_name}/chat/completions?api-version={inference_api_version}"
        
        payload = {
            "messages": test_messages  # No model field needed - it's in the URL
        }
        
        try:
            response = requests.post(
                chat_completions_url_openai,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            elif is_unsupported_operation_400(response):
                # Expected for embedding-only models invoked via chat/completions.
                print(f"  ℹ️  Backend returned 400 'operation unsupported' — expected for embedding-only models on chat/completions.")
                utils.print_ok(f"Model '{model_name}' - SUCCESS (routed; chat/completions not supported by this model)\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run Azure OpenAI API tests - missing API key, endpoint, or supported models")


👉🏽 
Testing models via Azure OpenAI API format...

👉🏽 Testing model: gpt-5.4
Response status: 200 - OK
  💬 Response: Paris
  📍 Backend Region: Italy North
✅ Model 'gpt-5.4' - SUCCESS
 ⌚ 14:48:58.068028 
👉🏽 Testing model: gpt-5.4-mini
Response status: 200 - OK
  💬 Response: Paris
  📍 Backend Region: Italy North
✅ Model 'gpt-5.4-mini' - SUCCESS
 ⌚ 14:49:00.088566 


<a id='test-sdk'></a>
### 🧪 Test using Azure OpenAI Python SDK

Test using the official Azure OpenAI Python SDK:

In [25]:
from openai import AzureOpenAI
import httpx

def strip_auth_header(request):
    """Remove Authorization header to prevent APIM JWT validation conflict.
    The AzureOpenAI SDK sends both api-key and Authorization: Bearer headers,
    but APIM tries to validate the Authorization header as a JWT token."""
    request.headers.pop("authorization", None)

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing with Azure OpenAI SDK using model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version,
            http_client=httpx.Client(
                event_hooks={"request": [strip_auth_header]}
            )
        )
        
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say 'Hello from Azure OpenAI SDK!'"}
            ]
        )
        
        utils.print_ok("SDK Test Successful!")
        print(f"💬 Response: {response.choices[0].message.content}")
        print(f"📊 Usage: {response.usage.total_tokens} tokens")
        
    except Exception as e:
        utils.print_error(f"SDK Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run SDK test - missing prerequisites")

👉🏽 Testing with Azure OpenAI SDK using model: gpt-5.4
✅ SDK Test Successful! ⌚ 14:49:15.178711 
💬 Response: Hello from Azure OpenAI SDK!
📊 Usage: 35 tokens


<a id='test-streaming'></a>
### 🧪 Test Streaming Response

Test streaming responses using the Azure OpenAI SDK:

In [26]:
from openai import AzureOpenAI
import httpx

def strip_auth_header(request):
    """Remove Authorization header to prevent APIM JWT validation conflict."""
    request.headers.pop("authorization", None)

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing streaming with model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version,
            http_client=httpx.Client(
                event_hooks={"request": [strip_auth_header]}
            )
        )
        
        start_time = time.time()
        
        response = client.chat.completions.with_raw_response.create(
            model=model_name,
            messages=[
                {"role": "user", "content": "Count from 1 to 10 with commas between each number."}
            ],
            stream=True
        )
        
        print(f"📡 x-ms-region: {response.headers.get('x-ms-region', 'unknown')}")
        print(f"📡 x-ms-stream: {response.headers.get('x-ms-stream', 'N/A')}")
        print("\n💬 Streaming response:")
        
        completion = response.parse()
        collected_content = []
        total_chunks = 0
        content_chunks = 0
        first_chunk_time = None
        
        for chunk in completion:
            total_chunks += 1
            if first_chunk_time is None:
                first_chunk_time = time.time()
            if chunk.choices and chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                collected_content.append(content)
                content_chunks += 1
                print(content, end='', flush=True)
        
        elapsed = time.time() - start_time
        ttfb = (first_chunk_time - start_time) if first_chunk_time else None
        print(f"\n\n✅ Stream completed in {elapsed:.2f} seconds")
        print(f"📦 Chunks received: {total_chunks} (with content: {content_chunks})")
        if ttfb is not None:
            print(f"⚡ Time to first chunk: {ttfb:.2f} seconds")
        print(f"📝 Full response: {''.join(collected_content)}")
        
    except Exception as e:
        utils.print_error(f"Streaming Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run streaming test - missing prerequisites")


👉🏽 Testing streaming with model: gpt-5.4
📡 x-ms-region: Italy North
📡 x-ms-stream: N/A

💬 Streaming response:
1, 2, 3, 4, 5, 6, 7, 8, 9, 10

✅ Stream completed in 2.68 seconds
📦 Chunks received: 31 (with content: 28)
⚡ Time to first chunk: 2.41 seconds
📝 Full response: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10


<a id='test-aliases'></a>
### 🧪 Test Model Aliases (consistent across Universal LLM, Azure OpenAI, Unified AI)

If you configured `model_aliases` in the initialization cell, this section sends a request using each **alias name** (instead of a real model) against:

- **Universal LLM API** — `model` field in the request body
- **Azure OpenAI API** — alias in the URL path (`/deployments/{alias}/chat/completions`)
- **Unified AI API** (if deployed) — wildcard route, `model` in body

The gateway's shared `resolve-model-alias` policy fragment translates each alias into a real underlying model based on the configured `priority` or `weighted` strategy. The cell below validates the resolution by inspecting the response payload and (when available) the `UAIG-Model-Id` debug header on Unified AI.

> Direct model selection still works unchanged — the alias resolver is a no-op when the requested model is not an alias name.


In [27]:
# Test that each configured model alias resolves correctly across both APIs.
# - Universal LLM API: alias is sent in the request body's `model` field.
# - Azure OpenAI API: alias is in the URL path (/deployments/{alias}/chat/completions).
# Both should return a successful chat completion. The response model name reflects
# the resolved underlying model (the gateway rewrites `model` in the body before forwarding).
#
# An alias that resolves to an embedding-only model on chat/completions will get HTTP 400
# "operation is unsupported" from the backend — that still proves the alias was resolved
# and routed correctly, so we treat it as a SUCCESSFUL routing test using
# `is_unsupported_operation_400` (defined in the per-model Universal LLM API test cell above).

if not model_aliases:
    utils.print_warning("No model aliases configured — skipping alias resolution tests.")
elif not api_key:
    utils.print_warning("No API key available — skipping alias resolution tests.")
else:
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "Reply with the single word OK."}
    ]

    utils.print_info(f"\nTesting {len(model_aliases)} model aliases via the Universal LLM API and Azure OpenAI API...\n")

    for alias in model_aliases:
        alias_name = alias["name"]
        strategy = alias.get("strategy", "priority")
        underlying = ", ".join(alias.get("models", []))
        weights = alias.get("weights")
        weights_str = f" weights={weights}" if weights else ""
        print(f"\n🔁 Alias: {alias_name}  (strategy: {strategy}{weights_str})")
        print(f"   → underlying models: [{underlying}]")

        # ---- Universal LLM API ----
        try:
            url_universal = f"{azure_endpoint_models}models/chat/completions?api-version={inference_api_version}"
            r = requests.post(
                url_universal,
                headers={"api-key": api_key},
                json={"model": alias_name, "messages": test_messages},
                timeout=60,
            )
            if r.status_code == 200:
                data = r.json()
                resolved_model = data.get("model", "?")
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "")
                utils.print_ok(f"   Universal LLM API → resolved model: '{resolved_model}'  reply: '{answer.strip()}'")
            elif is_unsupported_operation_400(r):
                utils.print_ok(f"   Universal LLM API → routed (alias resolved); backend returned 400 'operation unsupported' — expected for embedding-only target")
            else:
                utils.print_error(f"   Universal LLM API → {r.status_code}: {r.text[:200]}")
        except Exception as e:
            utils.print_error(f"   Universal LLM API → ERROR: {e}")

        # ---- Azure OpenAI API ----
        if azure_endpoint_openai:
            try:
                url_openai = f"{azure_endpoint_openai}openai/deployments/{alias_name}/chat/completions?api-version={inference_api_version}"
                r = requests.post(
                    url_openai,
                    headers={"api-key": api_key},
                    json={"messages": test_messages},
                    timeout=60,
                )
                if r.status_code == 200:
                    data = r.json()
                    resolved_model = data.get("model", "?")
                    answer = data.get("choices", [{}])[0].get("message", {}).get("content", "")
                    utils.print_ok(f"   Azure OpenAI API   → resolved model: '{resolved_model}'  reply: '{answer.strip()}'")
                elif is_unsupported_operation_400(r):
                    utils.print_ok(f"   Azure OpenAI API   → routed (alias resolved); backend returned 400 'operation unsupported' — expected for embedding-only target")
                else:
                    utils.print_error(f"   Azure OpenAI API   → {r.status_code}: {r.text[:200]}")
            except Exception as e:
                utils.print_error(f"   Azure OpenAI API   → ERROR: {e}")

        # ---- Unified AI API (optional; only if the API exists) ----
        try:
            apimClientTool.discover_api("unified-ai")
            azure_endpoint_unified = str(apimClientTool.azure_endpoint)
            url_unified = f"{azure_endpoint_unified}models/chat/completions?api-version={inference_api_version}"
            r = requests.post(
                url_unified,
                headers={"api-key": api_key, "UAIG-Config-Cache-Bypass": "true"},
                json={"model": alias_name, "messages": test_messages},
                timeout=60,
            )
            if r.status_code == 200:
                data = r.json()
                resolved_model = data.get("model", "?")
                uaig_model = r.headers.get("UAIG-Model-Id", "(header not enabled)")
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "")
                utils.print_ok(f"   Unified AI API     → resolved model: '{resolved_model}' (UAIG-Model-Id={uaig_model})  reply: '{answer.strip()}'")
            elif is_unsupported_operation_400(r):
                uaig_model = r.headers.get("UAIG-Model-Id", "(header not enabled)")
                utils.print_ok(f"   Unified AI API     → routed (UAIG-Model-Id={uaig_model}); backend returned 400 'operation unsupported' — expected for embedding-only target")
            else:
                utils.print_error(f"   Unified AI API     → {r.status_code}: {r.text[:200]}")
        except Exception as e:
            # API not deployed or other error — non-fatal for this validation
            utils.print_warning(f"   Unified AI API     → skipped: {str(e)[:120]}")


👉🏽 
Testing 2 model aliases via the Universal LLM API and Azure OpenAI API...


🔁 Alias: adv-gpt  (strategy: priority)
   → underlying models: [gpt-5.4]
✅    Universal LLM API → resolved model: 'gpt-5.4-2026-03-05'  reply: 'OK' ⌚ 14:49:32.982322 
✅    Azure OpenAI API   → resolved model: 'gpt-5.4-2026-03-05'  reply: 'OK' ⌚ 14:49:35.938887 
👉🏽 Found API with id /subscriptions/c25e83e9-00ef-4f10-8945-a9e113144476/resourceGroups/rg-citadel-italynorth-01/providers/Microsoft.ApiManagement/service/apim-82da66b0c8/apis/unified-ai-api and path unified-ai
👉🏽 Azure Endpoint with APIM https://apim-82da66b0c8.azure-api.net/
✅    Unified AI API     → resolved model: 'gpt-5.4-2026-03-05' (UAIG-Model-Id=(header not enabled))  reply: 'OK' ⌚ 14:49:39.479403 

🔁 Alias: mini-gpt  (strategy: priority)
   → underlying models: [gpt-5.4-mini]
✅    Universal LLM API → resolved model: 'gpt-5.4-mini-2026-03-17'  reply: 'OK' ⌚ 14:49:41.592556 
✅    Azure OpenAI API   → resolved model: 'gpt-5.4-mini-2026-03-17'  

---
## 📊 Results Summary
---

This notebook completed the following tasks:

1. **Extracted** current APIM backend-pools configurations
2. **Generated** a customizable LLM backend Terraform variables file (`terraform.tfvars`) supporting:
   - Backend types: `ai-foundry`, `azure-openai`, `aws-bedrock`, `aws-bedrock-mantle`, `gemini-openai`, `external`
   - Auth: legacy `auth_scheme` (`managedIdentity` | `apiKey` | `token`) and/or `auth_type` (`managed-identity`, `aws-sigv4`, `api-key-bearer`, `api-key-header`, `none`)
   - `auth_config` with optional **Key Vault references** (`key_vault_secret_uri`) for API-key backends
   - Per-model metadata (`sku`, `capacity`, `modelFormat`, `modelVersion`, `retirementDate`, `apiVersion`, `timeout`, `inferenceApiVersion`)
   - **Model aliases** (`model_aliases`) with `priority` or `weighted` strategies — same alias map honored consistently across Azure OpenAI, Universal LLM, and Unified AI APIs
   - **AWS credentials** (`aws_access_key_secret_uri`, `aws_secret_key_secret_uri` Key Vault references, `aws_region`) for Bedrock
3. **Deployed** via the standalone Terraform module ([`../llm-backend-onboarding`](../llm-backend-onboarding/README.md)) using `scripts/deploy.sh` (`terraform init` → import existing → `terraform apply`), which creates the shared `resolve-model-alias` policy fragment alongside backends, pools, and other fragments
4. **Verified** deployment including the `get-available-models` policy fragment
5. **Tested** the deployed models through:
   - **GET /deployments** (Microsoft Foundry integration — uses `get-available-models` policy fragment)
   - Universal LLM API (`/models/chat/completions`)
   - Azure OpenAI API (`/openai/deployments/{model}/chat/completions`)
   - Azure OpenAI Python SDK
   - Streaming responses
   - **Model alias resolution** across Universal LLM API, Azure OpenAI API, and Unified AI API (when configured)

### Next Steps

- Modify the `llm_backends_config` in the initialization cell to add more backends (Azure OpenAI, Bedrock, Bedrock Mantle, Gemini, external)
- For API-key backends, set `authConfig.keyVaultSecretUri` and `key_vault_name` to source secrets from Azure Key Vault (recommended over inline `secretValue`)
- Tune `model_aliases` to expose simplified client-facing names (e.g. `adv-gpt`, `fast-gpt`, `reasoning`) that route to one or more underlying models — `priority` strategy gives automatic cross-model fallback (Unified AI), `weighted` strategy enables A/B testing of model swaps
- Store the AWS keys in Key Vault and provide `aws_access_key_secret_uri` / `aws_secret_key_secret_uri` / `aws_region` when onboarding `aws-bedrock` (SigV4) backends
- Re-run the generate + deploy cells to update the APIM configuration, or run `./scripts/deploy.sh` directly from the [`../llm-backend-onboarding`](../llm-backend-onboarding/README.md) module
- Use the generated `terraform.tfvars` as a template for CI/CD pipelines (`terraform plan` / `terraform apply`)
- Use the `get-available-models` policy fragment to expose available models to Microsoft Foundry and other clients



<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

The backends and policy fragments deployed by this notebook are intended to persist as part of your Governance Hub configuration. To remove them, delete the corresponding backend resources and policy fragments from your APIM instance via the Azure portal or CLI.

> **Note:** Removing backends will affect any access contracts that reference those backend pools. Ensure no active contracts depend on them before cleanup.